<a href="https://colab.research.google.com/github/ticiAngelucci/GuiaLaboratorioQuantum/blob/main/modulo4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#Instalar Qiskit en el notebook
!pip install qiskit qiskit-aer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 58.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 99.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 73.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 4.3 MB/s eta 0:00:00


In [2]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

# Supongamos un estado 'secreto' preparado por el profesor
qc_secret = QuantumCircuit(1)
qc_secret.rx(np.pi/3, 0)
qc_secret.ry(np.pi/4, 0)

# Medición en las 3 bases: Z, X, Y
# Para X, rotamos con H. Para Y, rotamos con Sdg y H.
bases = ['z', 'x', 'y']
results = {}

for base in bases:
    qc = qc_secret.copy()
    if base == 'x':
        qc.h(0)
    elif base == 'y':
        qc.sdg(0); qc.h(0)
    qc.measure_all()

    sim = AerSimulator()
    job = sim.run(transpile(qc, sim), shots=2000)
    counts = job.result().get_counts()
    results[base] = counts

# Estimación (Inversión Lineal)
# E = (count_0 - count_1) / total
def get_expect(counts):
    total = sum(counts.values())
    return (counts.get('0', 0) - counts.get('1', 0)) / total

x = get_expect(results['x'])
y = get_expect(results['y'])
z = get_expect(results['z'])

print(f"Vector de Bloch estimado (r_x, r_y, r_z): ({x:.3f}, {y:.3f}, {z:.3f})")
print(f"Pureza estimada (Tr(rho^2)): {(1 + x**2 + y**2 + z**2)/2:.3f}")

Vector de Bloch estimado (r_x, r_y, r_z): (0.341, -0.880, 0.357)
Pureza estimada (Tr(rho^2)): 1.009


Notas:

La tomografía es la prueba de que en el mundo cuántico, la información es estadística. No medimos "el estado", medimos frecuencias relativas que convergen hacia la representación del estado.

¿Por qué la Tomografía requiere múltiples bases?
De acuerdo con el Principio de Incertidumbre, los operadores de Pauli no conmutan. Esto implica que no podemos conocer el vector de Bloch completo con una sola medición. La "incertidumbre" no es una deficiencia técnica de nuestro simulador, sino una restricción fundamental de la mecánica cuántica: un estado bien definido en la base Z debe tener una incertidumbre máxima en la base X. Por ello, la reconstrucción tomográfica (MLE) es necesaria para inferir el estado global a partir de proyecciones parciales.

Sección de Práctica: Desafíos de Laboratorio

Desafío 1:Inconsistencia Física

    Si aumentas el ruido del sistema (usando un NoiseModel), el vector reconstruido se encogerá hacia el origen (r<1). Explica cómo la pureza (Tr(ρ2)) es un indicador estadístico de cuánto ruido ha interactuado con nuestro sistema durante la preparación.

Desafío 2:MLE vs. Inversión Lineal

    Si el número de disparos (shots) es muy bajo, la inversión lineal puede dar resultados fuera de la esfera de Bloch. Investiga por qué el método MLE (máxima verosimilitud) es computacionalmente más costoso pero estadísticamente más robusto frente a fluctuaciones de muestreo.

Desafío 3:Matriz de Información

    La tomografía requiere medir el sistema en bases completas. Si solo midiéramos en la base Z y X, ¿qué información sobre el vector de Bloch perderíamos irrecuperablemente?